# 01 Data
Load raw BLUME data, check structure, coverage and drops

In [1]:
# Imports
import json                 # turns JSON text into Python dicts/lists
from pathlib import Path    # works with file paths

# Paths
ROOT = Path.cwd().parent    # cwd = folder the notebook runs in (notebooks/)
                            # project folder: notebook runs in notebooks/, go up one level
RAW = ROOT / "raw"          # raw API data: read only

print(RAW)
print(RAW.exists())

/Users/creative/Air_Quality_Project/raw
True


In [2]:
# Raw files, sorted so the order is the same on every run
files = sorted(RAW.glob("*.json"))

print(len(files))       # how many files?
for f in files[:5]:     # print the first 5 files
    print(f.name)       # just the file name, not the full path


324
2025-01_mc010_no2.json
2025-01_mc010_pm2.json
2025-01_mc027_no2.json
2025-01_mc032_no2.json
2025-01_mc032_pm2.json


In [3]:
# Read the first file into Python
data = json.loads(files[0].read_text()) # parse first raw file

print(files[0].name)       # file shown
print(type(data))          # type of data

2025-01_mc010_no2.json
<class 'list'>


In [4]:
print(len(data))  # number of entries
print(data[0])    # first entry
print(data[-1])   # last entry

744
{'datetime': '2025-02-01T00:00:00+01:00', 'station': 'mc010', 'core': 'no2', 'component': 'no2_1h', 'period': '1h', 'value': 24.0}
{'datetime': '2025-01-01T01:00:00+01:00', 'station': 'mc010', 'core': 'no2', 'component': 'no2_1h', 'period': '1h', 'value': 32.0}


**Findings**
- Expected hours per month in 2025: Jan (744), Feb (672), Mar (743, daylight saving starts), Apr (720), May (744), Jun (720), Jul (744), Aug (744), Sep (720), Oct (745, daylight saving ends), Nov (720), Dec (744). Year (8,760).
- January, mc010, NO₂: 744 hours. Complete.

In [5]:
# expected hours per month in 2025, Berlin time
EXPECTED = {
    "01": 744, "02": 672, "03": 743, "04": 720, "05": 744, "06": 720,
    "07": 744, "08": 744, "09": 720, "10": 745, "11": 720, "12": 744,
}

for f in files:
    month = f.name[5:7]                                  # "2025-01_mc010_no2.json" → "01"
    n = len(json.loads(f.read_text()))                   # entries in file
    if n != EXPECTED[month]:
        print(f.name, n, "expected", EXPECTED[month])    # incomplete files only

2025-01_mc174_no2.json 743 expected 744
2025-10_mc010_no2.json 744 expected 745
2025-10_mc010_pm2.json 744 expected 745
2025-10_mc027_no2.json 744 expected 745
2025-10_mc032_no2.json 744 expected 745
2025-10_mc032_pm2.json 744 expected 745
2025-10_mc042_no2.json 744 expected 745
2025-10_mc042_pm2.json 744 expected 745
2025-10_mc077_no2.json 744 expected 745
2025-10_mc077_pm2.json 744 expected 745
2025-10_mc085_no2.json 744 expected 745
2025-10_mc085_pm2.json 744 expected 745
2025-10_mc117_no2.json 744 expected 745
2025-10_mc117_pm2.json 744 expected 745
2025-10_mc124_no2.json 744 expected 745
2025-10_mc124_pm2.json 744 expected 745
2025-10_mc144_no2.json 744 expected 745
2025-10_mc144_pm2.json 744 expected 745
2025-10_mc145_no2.json 744 expected 745
2025-10_mc171_no2.json 744 expected 745
2025-10_mc171_pm2.json 744 expected 745
2025-10_mc174_no2.json 744 expected 745
2025-10_mc174_pm2.json 744 expected 745
2025-10_mc190_no2.json 744 expected 745
2025-10_mc190_pm2.json 744 expected 745


In [6]:
# October night of the clock change: check how the doubled hour is stored
oct_file = RAW / "2025-10_mc010_no2.json"
oct_data = json.loads(oct_file.read_text())

for entry in oct_data:
    if entry["datetime"].startswith("2025-10-26T0"):    # 26 Oct, 00:00–09:00
        print(entry["datetime"], entry["value"])

2025-10-26T09:00:00+01:00 9.0
2025-10-26T08:00:00+01:00 8.0
2025-10-26T07:00:00+01:00 7.0
2025-10-26T06:00:00+01:00 5.0
2025-10-26T05:00:00+01:00 2.0
2025-10-26T04:00:00+01:00 2.0
2025-10-26T03:00:00+01:00 2.0
2025-10-26T02:00:00+02:00 None
2025-10-26T01:00:00+02:00 6.0
2025-10-26T00:00:00+02:00 8.0


**Findings**
- October: the API skips the second 02:00 on 26 Oct (clock change). One hour missing in every file.
- Some entries have `None` as value: the hour exists, but nothing was measured. Entry counts alone don't show completeness.
- Timestamps include the UTC offset (+01:00 / +02:00), so converting to UTC gives a clean, gap-free timeline.

In [7]:
# missing values (None) per file
for f in files:
    data = json.loads(f.read_text())                            # entries in file
    n_none = sum(1 for entry in data if entry["value"] is None) # count entries with value None
    if n_none > 0:
        print(f.name, n_none)                                   # files with missing values only

2025-01_mc010_no2.json 2
2025-01_mc027_no2.json 1
2025-01_mc032_no2.json 1
2025-01_mc032_pm2.json 2
2025-01_mc042_no2.json 3
2025-01_mc077_no2.json 2
2025-01_mc085_no2.json 25
2025-01_mc085_pm2.json 24
2025-01_mc117_no2.json 2
2025-01_mc117_pm2.json 1
2025-01_mc124_no2.json 2
2025-01_mc124_pm2.json 1
2025-01_mc144_no2.json 3
2025-01_mc145_no2.json 2
2025-01_mc171_no2.json 3
2025-01_mc174_no2.json 3
2025-01_mc174_pm2.json 1
2025-01_mc190_no2.json 1
2025-01_mc221_no2.json 3
2025-01_mc282_no2.json 1
2025-02_mc010_no2.json 1
2025-02_mc144_no2.json 1
2025-02_mc144_pm2.json 1
2025-02_mc145_no2.json 1
2025-02_mc171_pm2.json 2
2025-02_mc282_no2.json 4
2025-03_mc010_no2.json 2
2025-03_mc010_pm2.json 2
2025-03_mc027_no2.json 2
2025-03_mc032_no2.json 3
2025-03_mc032_pm2.json 2
2025-03_mc042_no2.json 2
2025-03_mc042_pm2.json 1
2025-03_mc077_no2.json 2
2025-03_mc077_pm2.json 2
2025-03_mc085_no2.json 4
2025-03_mc085_pm2.json 56
2025-03_mc117_no2.json 2
2025-03_mc117_pm2.json 2
2025-03_mc124_no2.json

## Findings
- One file per month × station × pollutant. PM2.5 is named `pm2`.
- Timestamps include UTC offset. Newest first.
- All months complete, except mc174 NO₂ Jan (−1) and all October files (−1, clock change).
- Most files miss 1–6 values (`None`). Longer gaps: mc085 Jan, mc085 Mar, mc221 Apr, mc221 Jul, mc032 Dec.

## Decisions
- Missing values are not filled.
- Hours vs. 24-hour levels: indicator only.
- Days vs. 24-hour levels: only with ≥ 18 valid hours.

## Open checks
- Timestamp = end of hour?
- Days with < 18 valid hours?
- Missing hours at the same time of day?
- Fireworks hours present?